# Assignment 8 — Question 2: Spell Correction (Noisy Channel Model)

## Concept source: slp2.pdf §5.9 (The Noisy Channel Model for Spelling) & §5.9.1 (Contextual Spelling Error Correction)

The noisy channel model treats the observed (incorrect) word `w` as the output of a channel that corrupted the intended (correct) word `c`. We recover the intended word by maximizing:

P(c | w) ∝ P(w | c) × P(c)

- **P(c)** — language model: prior probability of the correct word (from `count_unigram.txt`).
- **P(w | c)** — channel / error model: probability that `c` was typed as `w` (from `count_1edit.txt`: 1-edit confusion counts).

§5.9.1 extends this to **contextual (real-word) spelling error correction**: when the error produces another valid English word (e.g. `there` for `three`, `dessert` for `desert`), surrounding context is used to disambiguate — i.e. condition on neighboring words, `P(c | context, w)`. We implement the core noisy channel below, then add a §5.9.1-style contextual rescorer.

**Data files (in `Spell-Correction/`):**
- `count_unigram.txt` — word → frequency (language model P(c))
- `count_1edit.txt` — `a|b` → count (1-edit confusion channel P(w|c))
- `spell-errors.txt` — `correct: err1, err2*2, ...` (test set)

In [1]:
import math
import re
from collections import defaultdict

UNIGRAM = "Spell-Correction/count_unigram.txt"
EDITS = "Spell-Correction/count_1edit.txt"
ERRORS = "Spell-Correction/spell-errors.txt"


def load_unigram(path):
    d = {}
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            w = parts[0].strip().lower()
            try:
                v = int(parts[1].strip())
            except ValueError:
                continue
            d[w] = v
    return d


def load_edits(path):
    d = {}
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            k = parts[0].strip()
            try:
                v = int(parts[1].strip())
            except ValueError:
                continue
            d[k] = v
    return d


def load_errors(path):
    """Return list of (correct, [err, ...]) expanding '*N' multipliers."""
    rows = []
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line or ":" not in line:
                continue
            left, right = line.split(":", 1)
            correct = left.strip().lower()
            errs = []
            for e in [x.strip() for x in right.split(",") if x.strip()]:
                m = re.match(r"([a-zA-Z]+)(?:\*(\d+))?$", e)
                if not m:
                    continue
                word = m.group(1).lower()
                mult = int(m.group(2)) if m.group(2) else 1
                errs.extend([word] * mult)
            if errs:
                rows.append((correct, errs))
    return rows


unigram = load_unigram(UNIGRAM)
edits = load_edits(EDITS)
errors = load_errors(ERRORS)
N_total = sum(unigram.values())

print("Unigram entries :", len(unigram))
print("Edit entries    :", len(edits))
print("Test items      :", len(errors))
print("Total error tokens :", sum(len(e) for _, e in errors))
print("N_total (unigram)  :", N_total)

Unigram entries : 333333
Edit entries    : 1585
Test items      : 7580
Total error tokens : 40598
N_total (unigram)  : 588124220187


In [2]:
                                                            
                                                                     
                                             
                                                         
                                                           
                                                                


def candidate_counts(err, edits_table):
    err = str(err)
    out = defaultdict(int)
    n = len(err)

    for k, c in edits_table.items():
        k = str(k)
        if "|" not in k:
            continue
        a, b = k.split("|", 1)
        if not a and not b:
            continue

                                                     
        if a and b:
            start = 0
            while True:
                pos = err.find(a, start)
                if pos == -1:
                    break
                cand = err[:pos] + b + err[pos + len(a):]
                out[cand] += c
                start = pos + len(a)

                                                   
        elif not a and b:
            for pos in range(n + 1):
                cand = err[:pos] + b + err[pos:]
                out[cand] += c

                                                  
        elif a and not b:
            start = 0
            while True:
                pos = err.find(a, start)
                if pos == -1:
                    break
                cand = err[:pos] + err[pos + len(a):]
                out[cand] += c
                start = pos + len(a)

    out[err] += 1                     
    return out


                                                      
channel_table = {}


def get_channel(err):
    if err not in channel_table:
        channel_table[err] = candidate_counts(err, edits)
    return channel_table[err]


                                                                
for err in ["rainning", "raning"]:
    cands = get_channel(err)
    print(f"\nerror = {err!r}  ({len(cands)} candidates)")
    top = sorted(cands.items(), key=lambda x: -x[1])[:8]
    for cand, cnt in top:
        print(f"   {cand!r:18s} channel_count={cnt}")


error = 'rainning'  (380 candidates)
   'reinning'         channel_count=856
   'raenning'         channel_count=771
   'rainneng'         channel_count=771
   'riinning'         channel_count=559
   'reainning'        channel_count=398
   'roinning'         channel_count=353
   'ralinning'        channel_count=353
   'raanning'         channel_count=313

error = 'raning'  (288 candidates)
   'rening'           channel_count=856
   'raneng'           channel_count=771
   'rining'           channel_count=559
   'reaning'          channel_count=398
   'roning'           channel_count=353
   'ralning'          channel_count=353
   'ranang'           channel_count=313
   'raming'           channel_count=230


In [3]:
                                                       

def log_channel(err, cand):
    v = get_channel(err).get(cand, 0)
    return math.log(v) if v > 0 else -20.0


def log_lang(cand):
    v = unigram.get(cand, 0)
    if v <= 0:
        return math.log(1.0) - math.log(N_total + 1)
    return math.log(v) - math.log(N_total)


def correct(err, top_k=10):
    """Return ranked list of (candidate, score) for an error."""
    cands = get_channel(err)
    scored = []
    for cand in cands:
        score = log_channel(err, cand) + log_lang(cand)
        scored.append((cand, score))
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]


             
for err, truth in [("rainning", "raining"), ("raning", "raining"), ("recieve", "receive")]:
    ranked = correct(err, top_k=5)
    print(f"\n{err!r} -> top corrections:")
    for cand, sc in ranked:
        mark = "  <-- CORRECT" if cand == truth else ""
        print(f"   {cand!r:18s} score={sc: .4f}{mark}")


'rainning' -> top corrections:
   'raining'          score=-8.4994  <-- CORRECT
   'rainning'         score=-12.9576
   'reinning'         score=-20.3479
   'raenning'         score=-20.4525
   'rainneng'         score=-20.4525

'raning' -> top corrections:
   'rating'           score=-4.9521
   'ranging'          score=-6.5422
   'raining'          score=-7.5738  <-- CORRECT
   'racing'           score=-8.0895
   'ranting'          score=-9.3480

'recieve' -> top corrections:
   'receive'          score=-4.4092  <-- CORRECT
   'recieved'         score=-7.6593
   'reciever'         score=-8.9457
   'recieve'          score=-9.2858
   'recieves'         score=-9.7536


In [4]:
                                             
total = 0
correct_top1 = 0
rank1 = rank3 = rank5 = rank10 = 0
not_in_table = 0
correct_rank_sum = 0
n_correct_ranked = 0
failures = []

for correct_word, errs in errors:
    for err in errs:
        total += 1
        ranked = correct(err, top_k=50)
        if not ranked:
            not_in_table += 1
            continue
        best = ranked[0][0]
        if best == correct_word:
            correct_top1 += 1
            rank1 += 1
                                     
        rank_of_correct = None
        for idx, (cand, _) in enumerate(ranked):
            if cand == correct_word:
                rank_of_correct = idx
                break
        if rank_of_correct is not None:
            n_correct_ranked += 1
            correct_rank_sum += (rank_of_correct + 1)
            if rank_of_correct < 3:
                rank3 += 1
            if rank_of_correct < 5:
                rank5 += 1
            if rank_of_correct < 10:
                rank10 += 1
        else:
            if len(failures) < 40:
                failures.append((err, correct_word, best))

print("=" * 60)
print("NOISY-CHANNEL SPELL CORRECTION RESULTS")
print("=" * 60)
print(f"total error tokens evaluated   : {total}")
print(f"top-1 correct                : {correct_top1}")
print(f"top-1 accuracy               : {correct_top1/total:.4f}")
print(f"rank-3 accuracy              : {rank3/total:.4f}")
print(f"rank-5 accuracy              : {rank5/total:.4f}")
print(f"rank-10 accuracy             : {rank10/total:.4f}")
print(f"correct form present in list : {n_correct_ranked/total:.4f}")
if n_correct_ranked:
    print(f"mean rank of correct (ranked): {correct_rank_sum/n_correct_ranked:.2f}")
print("=" * 60)
print("Sample failures (err -> correct ; predicted):")
for err, truth, pred in failures:
    print(f"   {err!r:18s} -> {truth!r:18s} ; pred {pred!r}")

NOISY-CHANNEL SPELL CORRECTION RESULTS
total error tokens evaluated   : 40598
top-1 correct                : 11870
top-1 accuracy               : 0.2924
rank-3 accuracy              : 0.3449
rank-5 accuracy              : 0.3570
rank-10 accuracy             : 0.3651
correct form present in list : 0.3701
mean rank of correct (ranked): 1.70
Sample failures (err -> correct ; predicted):
   'disparingly'      -> 'disparagingly'    ; pred 'disperingly'
   'forer'            -> 'four'             ; pred 'fore'
   'fuore'            -> 'four'             ; pred 'fuori'
   'fore'             -> 'four'             ; pred 'for'
   'fore'             -> 'four'             ; pred 'for'
   'fore'             -> 'four'             ; pred 'for'
   'fore'             -> 'four'             ; pred 'for'
   'fore'             -> 'four'             ; pred 'for'
   'haing'            -> 'hanging'          ; pred 'hang'
   'begining'         -> 'looking'          ; pred 'beginning'
   'luing'            -> 

## §5.9.1 Extension — Contextual Spelling Error Correction

(*Contextual Spelling Error Correction*) handles **real-word errors**: typos that accidentally form another valid English word (e.g. `there` for `three`, `dessert` for `desert`, `piece` for `peace`). Because both the error and the correction are real words, the channel model alone is insufficient; we must condition on context:

P(c | context, w) ∝ P(w | c) × P(c | context)

Here `P(c | context)` is a higher-order language model (e.g. bigram) that scores how likely `c` is given its neighboring words.

Below is a demonstrator of the §5.9.1 mechanism. Since the provided files only supply unigram counts (no bigram table), `P(c | context)` is approximated with the unigram prior plus a simple context-compatibility bonus; in a full system you would replace this with a trained bigram/neural LM.

In [5]:
                                                                                 


def contextual_score(err, cand, left=None, right=None):
    """Noisy-channel score boosted by unigram-based context compatibility."""
    base = log_channel(err, cand) + log_lang(cand)
    ctx_bonus = 0.0
                                                                                       
    if cand in unigram:
        ctx_bonus += math.log(unigram[cand]) - math.log(N_total)
                                                  
    if left and cand == left:
        ctx_bonus -= 5.0
    if right and cand == right:
        ctx_bonus -= 5.0
    return base + ctx_bonus


def contextual_correct(err, left=None, right=None, top_k=5):
    cands = get_channel(err)
    scored = [(c, contextual_score(err, c, left, right)) for c in cands]
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]


print("§5.9.1 contextual demo (real-word style errors):")
demos = [
    ("dessert", "desert", None, None),
    ("peace", "piece", None, None),
    ("there", "three", None, None),
    ("recieve", "receive", None, None),
]
for err, truth, left, right in demos:
    ranked = contextual_correct(err, left, right, top_k=5)
    print(f"\n{err!r} (true={truth!r}):")
    for cand, sc in ranked:
        mark = "  <-- CORRECT" if cand == truth else ""
        print(f"   {cand!r:16s} score={sc: .4f}{mark}")

print()
print("Note: a production §5.9.1 system would train P(c | context) from a bigram/")
print("neural language model and use it in place of the unigram prior above.")

§5.9.1 contextual demo (real-word style errors):

'dessert' (true='desert'):
   'desert'         score=-16.1452  <-- CORRECT
   'desserts'       score=-19.5597
   'dessert'        score=-19.6777
   'dissert'        score=-20.2791
   'dessirt'        score=-20.2791

'peace' (true='piece'):
   'place'          score=-14.1639
   'peace'          score=-15.1441
   'pace'           score=-17.2020
   'peaci'          score=-20.2791
   'peece'          score=-20.3479

'there' (true='three'):
   'there'          score=-10.0966
   'three'          score=-10.5096  <-- CORRECT
   'these'          score=-11.3435
   'where'          score=-13.6960
   'theme'          score=-16.9378

'recieve' (true='receive'):
   'receive'        score=-13.2128  <-- CORRECT
   'ricieve'        score=-20.2791
   'reciive'        score=-20.2791
   'recievi'        score=-20.2791
   'receeve'        score=-20.4525

Note: a production §5.9.1 system would train P(c | context) from a bigram/
neural language model and use